# Крок 1. Огляд даних

Перед побудовою моделі потрібно зафіксувати склад і якість доступних даних та визначити межі
допустимого використання атрибутів. Це знижує ризик методологічних помилок, насамперед —
потрапляння до ознак інформації, що стає відомою лише після публікації.

Ключове обмеження задачі: рішення приймається до публікації. Відповідно, метрики залученості
(перегляди, вподобання, коментарі, поширення, збереження) є показниками пост-фактум і
використовуються виключно для формування цільової змінної, а не як вхідні ознаки.

Нижче послідовно розглянуто обидва запропоновані джерела, оцінено їх придатність та, за потреби,
дібрано альтернативу.

## 1. Джерело №1 — `datahiveai/Tiktok-Videos`

In [1]:
from pathlib import Path
import urllib.request
import numpy as np
import pandas as pd

# уніфікація робочого каталогу: корінь репозиторію або тека notebooks/
ROOT = Path.cwd(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
raw_path = ROOT / "data" / "raw" / "train.csv"
if not raw_path.exists():                      # одноразове завантаження з кешуванням
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    url = "https://huggingface.co/datasets/datahiveai/Tiktok-Videos/resolve/main/train.csv"
    print("завантаження джерела №1…"); urllib.request.urlretrieve(url, raw_path)

df1 = pd.read_csv(raw_path)
print("розмір:", df1.shape)
print("кількість авторів (author_unique_id):", df1["author_unique_id"].nunique())
print("атрибути:", list(df1.columns))
df1.head(3)

розмір: (2060, 14)
кількість авторів (author_unique_id): 4
атрибути: ['url', 'digg_count', 'play_count', 'share_count', 'repost_count', 'collect_count', 'comment_count', 'video_id', 'author_id', 'duration', 'description', 'create_time', 'author_unique_id', 'location_created']


,url,digg_count,play_count,share_count,repost_count,collect_count,comment_count,video_id,author_id,duration,description,create_time,author_unique_id,location_created
0,https://www.tiktok.com/@zachking/video/1001169...,857800.0,1700000.0,476.0,0.0,335.0,1508.0,100116967235219456,6.861650e+16,0.0,When it's trash night at my house #dailylife,1464212460,zachking,NaN
1,https://www.tiktok.com/@zachking/video/1164457...,1100000.0,2000000.0,637.0,0.0,467.0,1988.0,116445712837398528,6.861650e+16,15.0,#MamaSaid to always be a gentleman,1468105536,zachking,NaN
2,https://www.tiktok.com/@zachking/video/1165721...,1000000.0,2100000.0,790.0,0.0,455.0,2683.0,116572195421646848,6.861650e+16,11.0,I've got #NoMoney ...oh wait,1468135692,zachking,NaN


In [2]:
# базова діагностика якості: типи, повнота, відомі особливості джерела
print("типи:"); print(df1.dtypes)
print("\nчастка пропусків (за спаданням):")
print(df1.isna().mean().round(3).sort_values(ascending=False).head(6))
print("\nrepost_count тотожно нульовий:", bool((df1["repost_count"].fillna(0) == 0).all()))
ct = pd.to_datetime(df1["create_time"], unit="s", utc=True)
print("діапазон create_time:", ct.min().date(), "—", ct.max().date())
print("\nрозподіл відео за авторами:")
print(df1["author_unique_id"].value_counts())

типи:
url                  object
digg_count          float64
play_count          float64
share_count         float64
repost_count        float64
collect_count       float64
comment_count       float64
video_id              int64
author_id           float64
duration            float64
description          object
create_time           int64
author_unique_id     object
location_created     object
dtype: object

частка пропусків (за спаданням):
description         0.044
location_created    0.030
digg_count          0.001
play_count          0.001
share_count         0.001
repost_count        0.001
dtype: float64

repost_count тотожно нульовий: True
діапазон create_time: 1970-01-01 — 2025-02-27

розподіл відео за авторами:
author_unique_id
williesalim    1008
zachking        481
mrbeast         347
addisonre       221
Name: count, dtype: int64


In [3]:
# описова статистика метрик залученості та тривалості (масштаб, асиметрія розподілів)
df1[["play_count","digg_count","share_count","comment_count","collect_count","duration"]].describe().round(1)

,play_count,digg_count,share_count,comment_count,collect_count,duration
count,2.057000e+03,2057.0,2057.0,2057.0,2057.0,2057.0
mean,2.172212e+07,1568597.5,33166.0,24734.4,57167.1,47.0
std,4.384796e+07,2320108.2,258133.2,120100.9,101476.0,38.8
min,3.270000e+05,19900.0,299.0,272.0,203.0,0.0
25%,5.900000e+06,360100.0,3494.0,3257.0,11430.0,15.0
50%,1.260000e+07,832900.0,8414.0,7498.0,26306.0,40.0
75%,2.430000e+07,1800000.0,20100.0,18900.0,62686.0,70.0
max,1.100000e+09,25400000.0,10200000.0,4800000.0,1360762.0,622.0


### Спостереження
Атрибути поділяються на доступні до публікації (`description`, `duration`, `create_time`, автор —
кандидати в ознаки) та пост-фактум (лічильники — лише для цільової змінної); `url` і `video_id` є
ідентифікаторами. Ліцензія — CC BY-NC 4.0 (некомерційне використання).

Основне обмеження: вибірка містить лише чотирьох авторів. Це знижує репрезентативність і не дає
підстав узагальнювати результати на ширшу сукупність авторів. Крім того, абсолютна кількість
переглядів значною мірою відображає популярність автора, а не якість окремого відео, що слід
враховувати при визначенні цільової змінної.

Додатково: `repost_count` тотожно нульовий (виключається); часовий діапазон охоплює кілька років,
що вказує на дрейф розподілів і потребує врахування при розбитті; ознаки контенту (відео, аудіо)
відсутні.

Джерело придатне для формування цільової змінної, проте обмежене за обсягом і репрезентативністю.

## 2. Джерело №2 — `wasifullahcs/tiktok-video-dataset` (Kaggle)

Завантаження потребує автентифікації; склад полів встановлено за публічними метаданими.

In [4]:
import pandas as pd
# перелік полів за офіційними метаданими набору
schema2 = pd.DataFrame([
    ("aweme_id",   "ідентифікатор", "ідентифікатор відео"),
    ("video_id",   "ідентифікатор", "альтернативний ідентифікатор"),
    ("region",     "до публікації", "регіон"),
    ("keyword",    "до публікації", "пошуковий запит / категорія"),
    ("title",      "до публікації", "назва / підпис"),
    ("cover",      "медіа-URL",     "зображення обкладинки"),
    ("duration",   "до публікації", "тривалість (с)"),
    ("play_url",   "медіа-URL",     "посилання на відео"),
    ("wmplay_url", "медіа-URL",     "відео з водяним знаком"),
    ("size",       "медіа",         "розмір файлу"),
    ("wm_size",    "медіа",         "розмір версії з водяним знаком"),
    ("music_url",  "медіа-URL",     "аудіодоріжка"),
    ("timestamp",  "метадані",      "час збору даних"),
], columns=["поле", "категорія", "опис"])
print("Джерело №2: 13 полів, формат CSV, ліцензія MIT.")
print("наявні метрики залученості:", any(f in schema2["поле"].values for f in ["play_count","digg_count","like_count"]))
print("наявний ідентифікатор автора:", any(("author" in f or "creator" in f) for f in schema2["поле"]))
schema2

Джерело №2: 13 полів, формат CSV, ліцензія MIT.
наявні метрики залученості: False
наявний ідентифікатор автора: False


,поле,категорія,опис
0,aweme_id,ідентифікатор,ідентифікатор відео
1,video_id,ідентифікатор,альтернативний ідентифікатор
2,region,до публікації,регіон
3,keyword,до публікації,пошуковий запит / категорія
4,title,до публікації,назва / підпис
5,cover,медіа-URL,зображення обкладинки
6,duration,до публікації,тривалість (с)
7,play_url,медіа-URL,посилання на відео
8,wmplay_url,медіа-URL,відео з водяним знаком
9,size,медіа,розмір файлу


### Оцінка придатності
Джерело не відповідає вимогам задачі. Відсутні метрики залученості, що унеможливлює формування
цільової змінної та супервізовану постановку. Відсутній ідентифікатор автора, що унеможливлює
оцінку результату відносно автора. Відсутній спільний ключ із джерелом №1, що унеможливлює
інтеграцію. Потенційну цінність має лише сирий медіаконтент (посилання на відео й аудіо), проте за
відсутності цільової змінної його опрацювання виходить за межі поточного обсягу робіт. Джерело
виключається з подальшого розгляду.

## 3. Підбір альтернативного джерела

Критерії відбору: наявність метрик залученості (для цільової змінної); достатня кількість авторів
(репрезентативність, можливість оцінки відносно автора); наявність атрибутів, відомих до публікації;
прийнятні обсяг і ліцензія. Розглянуті кандидати:

| Джерело | Обсяг | Автори | Метрики залученості | Висновок |
|---|---|---|---|---|
| **lingbow/tiktok-video-engagement-200k** | 210 000 | багато | наявні (`engagement_daily`) | обрано |
| yakhyojon/tiktok (CC0) | 19 000 | відсутні (лише статуси) | наявні | резервне; без автора та часу |
| tarekmasryo/youtube-tiktok-trends-2025 | 50 000 | відсутні | лише похідні показники | ймовірно синтетичне; відхилено |
| haseebindata/...performance | 5 | — | наявні | непридатне за обсягом |

Обрано джерело `lingbow`: воно усуває основне обмеження джерела №1 (концентрацію на чотирьох
авторах), містить значну кількість авторів та атрибути, відомі до публікації. Нижче перевірено
доступність джерела та відповідність його структури вимогам.

In [5]:
import os, warnings, logging
# середовище без HF-токена та поза Jupyter-віджетами видає нерелевантні попередження
# (tqdm/IProgress, unauthenticated HF); глушимо їх, щоб не засмічували вивід
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
for _n in ("datasets", "huggingface_hub", "huggingface_hub.utils._http", "fsspec"):
    logging.getLogger(_n).setLevel(logging.ERROR)

from IPython.display import display
import pandas as pd
from pathlib import Path

def _head(name, n=2000):
    for base in ("data/raw/lingbow", "../data/raw/lingbow"):
        p = Path(base) / f"{name}.parquet"
        if p.exists():
            return pd.read_parquet(p).head(n)
    import itertools
    from datasets import load_dataset
    ds = load_dataset("lingbow/tiktok-video-engagement-200k", name, split="train", streaming=True)
    return pd.DataFrame(list(itertools.islice(ds, n)))
vdf = _head("videos"); edf = _head("engagement_daily")

# категорія + опис за відомими полями (структуру подаємо таблицею, як для джерел №1–2)
_meta = {
    "video_id": ("ідентифікатор", "ідентифікатор відео"),
    "author_id": ("ідентифікатор", "ідентифікатор автора"),
    "create_time": ("до публікації", "час створення"),
    "create_date": ("до публікації", "дата створення"),
    "duration": ("до публікації", "тривалість (с)"),
    "ratio": ("до публікації", "співвідношення сторін"),
    "desc": ("до публікації · текст", "підпис"),
    "desc_language": ("до публікації · текст", "мова підпису"),
    "is_english": ("до публікації · текст", "ознака англійської мови"),
    "sticker_text": ("до публікації · текст", "текст стікерів"),
    "hashtags": ("до публікації · текст", "перелік хештегів"),
    "transcript": ("до публікації · похідне (LLM)", "транскрипт мовлення"),
    "word_count": ("до публікації · похідне", "кількість слів"),
    "emoji_count": ("до публікації · похідне", "кількість емодзі"),
    "question_count": ("до публікації · похідне", "кількість запитань"),
    "hashtag_count": ("до публікації · похідне", "кількість хештегів"),
    "speaking_rate": ("до публікації · похідне", "темп мовлення"),
    "gpt_summary": ("до публікації · похідне (LLM)", "стислий зміст"),
    "topic": ("до публікації · похідне (LLM)", "тема"),
    "anger": ("до публікації · похідне (LLM)", "оцінка емоції: гнів"),
    "joy": ("до публікації · похідне (LLM)", "оцінка емоції: радість"),
    "surprise": ("до публікації · похідне (LLM)", "оцінка емоції: подив"),
    "sadness": ("до публікації · похідне (LLM)", "оцінка емоції: сум"),
    "disgust": ("до публікації · похідне (LLM)", "оцінка емоції: відраза"),
    "fear": ("до публікації · похідне (LLM)", "оцінка емоції: страх"),
    "music_id": ("до публікації · музика", "ідентифікатор треку"),
    "music_title": ("до публікації · музика", "назва треку"),
    "music_author": ("до публікації · музика", "виконавець"),
    "music_album": ("до публікації · музика", "альбом"),
    "music_owner_id": ("до публікації · музика", "власник треку"),
    "music_selected_from": ("до публікації · музика", "джерело вибору музики"),
    "created_by_ai": ("службове", "ознака згенерованого ШІ"),
    "is_ads": ("службове", "ознака реклами"),
    "date": ("службове", "дата зрізу"),
    "days_since_post": ("службове", "днів від публікації"),
    "play_count": ("пост-фактум", "перегляди (накопичувально)"),
    "like_count": ("пост-фактум", "вподобання"),
    "comment_count": ("пост-фактум", "коментарі"),
    "share_count": ("пост-фактум", "поширення"),
    "collect_count": ("пост-фактум", "збереження"),
    "download_count": ("пост-фактум", "завантаження"),
    "whatsapp_share_count": ("пост-фактум", "поширення у WhatsApp"),
}
def _schema(df):
    rows = [(c, *_meta.get(c, ("інше", ""))) for c in df.columns]
    return pd.DataFrame(rows, columns=["поле", "категорія", "опис"])

videos_schema, engagement_schema = _schema(vdf), _schema(edf)
print("videos:", vdf.shape[1], "атрибутів | унікальних авторів у підвибірці 2000:", vdf["author_id"].nunique())
# характер лічильників (накопичувальні чи добові) визначає спосіб формування цільової змінної
g = edf.sort_values("days_since_post").groupby("video_id")["play_count"].apply(list).iloc[0]
print("динаміка play_count за днями:", g[:6], "→ монотонне зростання (накопичувальні)")
print("\nСтруктура `videos`:"); display(videos_schema)
print("Структура `engagement_daily`:"); display(engagement_schema)

videos: 33 атрибутів | унікальних авторів у підвибірці 2000: 220
динаміка play_count за днями: [1737, 1772, 1778, 1788, 1789, 1791] → монотонне зростання (накопичувальні)

Структура `videos`:


,поле,категорія,опис
0,video_id,ідентифікатор,ідентифікатор відео
1,author_id,ідентифікатор,ідентифікатор автора
2,create_time,до публікації,час створення
3,create_date,до публікації,дата створення
4,duration,до публікації,тривалість (с)
5,ratio,до публікації,співвідношення сторін
6,desc_language,до публікації · текст,мова підпису
7,is_english,до публікації · текст,ознака англійської мови
8,desc,до публікації · текст,підпис
9,sticker_text,до публікації · текст,текст стікерів


Структура `engagement_daily`:


,поле,категорія,опис
0,video_id,ідентифікатор,ідентифікатор відео
1,date,службове,дата зрізу
2,days_since_post,службове,днів від публікації
3,play_count,пост-фактум,перегляди (накопичувально)
4,like_count,пост-фактум,вподобання
5,comment_count,пост-фактум,коментарі
6,share_count,пост-фактум,поширення
7,collect_count,пост-фактум,збереження
8,download_count,пост-фактум,завантаження
9,whatsapp_share_count,пост-фактум,поширення у WhatsApp


### Обґрунтування вибору (за результатами перевірки)
Перевірка підтверджує відповідність джерела критеріям відбору:
- **кількість авторів** — у підвибірці з 2000 записів виявлено близько 220 унікальних авторів, що
  на повному обсязі відповідає тисячам авторів; обмеження джерела №1 (концентрація на чотирьох
  авторах) усунуто;
- **атрибути до публікації** — наявні підпис (`desc`), хештеги, тривалість, час створення, а також
  похідні текстові ознаки (транскрипт, лічильники слів/емодзі, темп мовлення) та метадані музики;
- **метрики залученості** — наявні у `engagement_daily` (перегляди, вподобання, коментарі,
  поширення, збереження, завантаження) і є накопичувальними, що дозволяє коректно визначити
  підсумковий результат відео;
- **обсяг і ліцензія** — обсяг достатній (підлягає підвибірці); ліцензія CC-BY-NC прийнятна для
  некомерційного дослідницького використання.

Сукупно джерело задовольняє всі критерії та обирається як основне.

### Передумови постановки задачі
Лічильники у `engagement_daily` накопичувальні (підтверджено емпірично), тому підсумковий результат
відео визначається записом із максимальним `days_since_post` і приєднується до таблиці `videos` за
ключем `video_id`. Цільову змінну доцільно визначати відносно автора (перевищення власної медіани),
що зменшує вплив відмінностей у популярності авторів.

Наявність `days_since_post` додатково дозволяє використати дані першого дня як ранній сигнал; отже
джерело підтримує як сценарій до публікації, так і сценарій за наявності початкових даних.

Обмеження та ризики: ліцензія CC-BY-NC (некомерційне використання); значний обсяг (210 000 відео та
кілька мільйонів записів залученості) потребує підвибірки; атрибути `gpt_summary` та оцінки емоцій є
похідними мовної моделі (походження фіксується, використання необов'язкове); історія автора
(`creator_daily`) застосовується виключно станом до публікації для запобігання витоку даних.

Джерело №1 зберігається як допоміжне для перехресної перевірки на іншому зрізі.

### Відкриті питання
Підлягають узгодженню: сценарій прогнозування (до публікації / за наявності даних першого дня);
визначення цільової змінної (відносно автора / глобальний поріг); архітектура рішення та формат
користувацького інтерфейсу.